## Cortical Thickness — Prodromal Subgroup Analysis

Compares **RBD** and **Hyposmia** subgroups against **Healthy Controls** using NiSpace.

- Contrasts: RBD vs HC · Hyposmia vs HC
- Phenotype: Cortical thickness (Desikan–Killiany, 68 parcels)
- Subgroup IDs: RBD = 5.0, Hyposmia = 6.0, HC = 2.0 (remapped from `subgroup` column)
- Same reference maps, covariates, and helper functions as `thickness_cortical_shi.ipynb`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

np.random.seed(42)

from nispace.datasets import fetch_reference
from nispace.plotting import view_surf
from nispace.workflows import group_comparison
import statsmodels.api as sm

In [ ]:
DATA_PATH = Path("../../data/df1.csv")

DK_REGIONS = [
    "bankssts","caudalanteriorcingulate","caudalmiddlefrontal","cuneus","entorhinal",
    "fusiform","inferiorparietal","inferiortemporal","isthmuscingulate","lateraloccipital",
    "lateralorbitofrontal","lingual","medialorbitofrontal","middletemporal","parahippocampal",
    "paracentral","parsopercularis","parsorbitalis","parstriangularis","pericalcarine",
    "postcentral","posteriorcingulate","precentral","precuneus","rostralanteriorcingulate",
    "rostralmiddlefrontal","superiorfrontal","superiorparietal","superiortemporal",
    "supramarginal","frontalpole","temporalpole","transversetemporal","insula",
]

DEMO_COLS = ["PATNO", "CONCOHORT", "age", "SEX", "agediag", "subgroup", "PRIMDIAG"]

# Subgroup group IDs (remapped CONCOHORT)
SUBGROUP_TO_ID = {"Healthy Control": 2.0, "RBD": 5.0, "Hyposmia": 6.0}

CONTRASTS = {
    "RBD vs HC": (5.0, 2.0),
    "Hyposmia vs HC": (6.0, 2.0),
}

GROUP_LABELS = {5.0: "RBD", 6.0: "Hyposmia", 2.0: "HC"}

SELECTED_REFERENCE_MAPS = [
    "target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019",
    "target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021",
    "target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018",
    "target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018",
    "target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017",
    "target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015",
    "target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018",
    "target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012",
    "target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012",
    "target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012",
    "target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017",
    "target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012",
    "target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017",
    "target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017",
]

N_PERM = 10000

In [ ]:
# ── Helper functions (identical to thickness_cortical_shi.ipynb) ──────────────

def get_dk_thickness_columns(df):
    dk_set = {r.lower() for r in DK_REGIONS}
    result = []
    for col in df.columns:
        col_lower = col.lower()
        if not col_lower.endswith("_thickness"):
            continue
        for hemi in ("lh_", "rh_"):
            if col_lower.startswith(hemi):
                region = col_lower[len(hemi):-len("_thickness")]
                if region in dk_set:
                    result.append(col)
                break
    return result

def rename_to_nispace(col):
    match = re.match(r"(lh|rh)_(.+)_thickness", col)
    if not match:
        return col
    hemi, region = match.groups()
    hemi_letter = "L" if hemi == "lh" else "R"
    return f"hemi-{hemi_letter}_lab-{region}"

def prepare_brain_and_design(df):
    demo_cols = [c for c in DEMO_COLS if c in df.columns]
    dk_cols = get_dk_thickness_columns(df)
    print("Number of DK thickness columns:", len(dk_cols))
    extra_cols = [c for c in ["Field Strength"] if c in df.columns]
    df_dk = df[demo_cols + extra_cols + dk_cols].copy()
    Y = df_dk[dk_cols].copy()
    Y.index = df_dk["PATNO"]
    design = df_dk[["CONCOHORT", "age", "SEX", "Field Strength"]].copy()
    design.index = df_dk["PATNO"]
    design["CONCOHORT"] = pd.to_numeric(design["CONCOHORT"], errors="coerce")
    design["Field Strength"] = pd.to_numeric(design["Field Strength"], errors="coerce")
    return Y, design

def align_y_to_reference(Y, ref_df):
    Y_renamed = Y.copy()
    Y_renamed.columns = [rename_to_nispace(c) for c in Y.columns]
    Y_aligned = Y_renamed.reindex(columns=ref_df.columns).copy()
    print("Y aligned shape:", Y_aligned.shape)
    return Y_aligned

def run_parcelwise_ttest(Y_aligned, design, g1, g2):
    mask = design["CONCOHORT"].astype(float).isin([g1, g2])
    y_sub = Y_aligned.loc[mask].copy()
    d_sub = design.loc[mask].copy()
    d_sub["group01"] = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1})
    if d_sub["SEX"].dtype == object:
        d_sub["SEX"] = pd.Categorical(d_sub["SEX"]).codes
    d_sub["SEX"] = pd.to_numeric(d_sub["SEX"], errors="coerce")
    d_sub["age"] = pd.to_numeric(d_sub["age"], errors="coerce")
    d_sub["Field Strength"] = pd.to_numeric(d_sub["Field Strength"], errors="coerce")
    t_vals, p_vals, dfs = [], [], []
    for parcel in y_sub.columns:
        tmp = pd.DataFrame({
            "y": pd.to_numeric(y_sub[parcel], errors="coerce"),
            "group01": d_sub["group01"], "age": d_sub["age"],
            "SEX": d_sub["SEX"], "Field Strength": d_sub["Field Strength"],
        }, index=y_sub.index).dropna()
        if tmp.shape[0] < 5 or tmp["group01"].nunique() < 2:
            t_vals.append(np.nan); p_vals.append(np.nan); dfs.append(np.nan)
            continue
        X = sm.add_constant(tmp[["group01", "age", "SEX", "Field Strength"]])
        model = sm.OLS(tmp["y"], X).fit()
        t_vals.append(model.tvalues.get("group01", np.nan))
        p_vals.append(model.pvalues.get("group01", np.nan))
        dfs.append(model.df_resid)
    return pd.DataFrame(
        {"Tvalue": t_vals, "pvalue": p_vals, "df": dfs,
         "hemi": ["L" if c.startswith("hemi-L") else "R" for c in Y_aligned.columns]},
        index=Y_aligned.columns,
    )

def build_nispace_design(d_sub, g1, g2):
    groups01 = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1}).astype(int)
    design_df = pd.DataFrame({
        "groups": groups01,
        "age": pd.to_numeric(d_sub["age"], errors="coerce"),
        "SEX": d_sub["SEX"],
        "Field Strength": d_sub["Field Strength"],
    }, index=d_sub.index)
    if design_df["SEX"].dtype == object:
        design_df["SEX"] = pd.Categorical(design_df["SEX"]).codes
    design_df["SEX"] = pd.to_numeric(design_df["SEX"], errors="coerce")
    design_df["Field Strength"] = pd.to_numeric(design_df["Field Strength"], errors="coerce")
    design_df["Field Strength"] = design_df["Field Strength"].map({1.5: 0, 3.0: 1})
    return design_df

def run_group_comparisons(Y_aligned, design, ref_df, contrasts, labels, n_perm=10000):
    all_rows = []
    outputs = {}
    for contrast_name, (g1, g2) in contrasts.items():
        mask = design["CONCOHORT"].astype(float).isin([g1, g2])
        y_sub = Y_aligned.loc[mask].copy()
        d_sub = design.loc[mask].copy()
        design_df = build_nispace_design(d_sub, g1, g2)
        keep = ~(y_sub.isna().any(axis=1) | design_df[["groups","age","SEX","Field Strength"]].isna().any(axis=1))
        y_sub = y_sub.loc[keep]; design_df = design_df.loc[keep]
        print(f"\nContrast: {contrast_name}")
        print("Counts:", d_sub.loc[keep, "CONCOHORT"].astype(float).map(labels).value_counts().to_dict())
        np.random.seed(42)
        colocs, pvals, qvals, nsp = group_comparison(
            y=y_sub, x=ref_df, parcellation="DesikanKilliany", design=design_df,
            comparison_method="hedges(a,b)", colocalization_method="spearman",
            n_perm=n_perm, n_proc=-1, verbose=True,
        )
        outputs[contrast_name] = {"colocs": colocs, "p": pvals, "q": qvals, "nsp": nsp}
        df_out = pd.DataFrame({"reference_map": ref_df.index, "rho": np.asarray(colocs).ravel(),
                               "p": np.asarray(pvals).ravel(), "q": np.asarray(qvals).ravel(),
                               "contrast": contrast_name})
        all_rows.append(df_out)
    return pd.concat(all_rows, ignore_index=True).sort_values(["contrast","q","p"]), outputs

In [ ]:
# ── Load data and remap subgroup → CONCOHORT ──────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

# Filter to relevant subgroups only
df = df_raw[df_raw["subgroup"].isin(SUBGROUP_TO_ID.keys())].copy()
df["CONCOHORT"] = df["subgroup"].map(SUBGROUP_TO_ID)

print("Subgroup counts:")
print(df["subgroup"].value_counts())
print("\nCONCOHORT after remapping:")
print(df["CONCOHORT"].value_counts())
df.head(3)

In [ ]:
Y, design = prepare_brain_and_design(df)
print("Y shape:", Y.shape)
print("Design shape:", design.shape)

In [ ]:
df_reference_desikan = fetch_reference(
    "pet", collection="UniqueTracers", parcellation="DesikanKilliany", print_references=True,
)
df_reference_selected = df_reference_desikan[
    df_reference_desikan.index.get_level_values("map").isin(SELECTED_REFERENCE_MAPS)
]
print("Selected reference shape:", df_reference_selected.shape)

In [ ]:
Y_aligned = align_y_to_reference(Y, df_reference_selected)
common_idx = Y_aligned.index.intersection(design.index)
Y_aligned = Y_aligned.loc[common_idx].copy()
design = design.loc[common_idx].copy()

In [ ]:
# ── Parcelwise t-tests ────────────────────────────────────────────────────────
parcelwise_ttests = {}
for contrast_name, (g1, g2) in CONTRASTS.items():
    df_ttest = run_parcelwise_ttest(Y_aligned, design, g1, g2)
    df_ttest_reset = df_ttest.reset_index().rename(columns={"index": "parcel"})
    safe_name = contrast_name.lower().replace(" ", "_")
    outname = f"../../results/{safe_name}_parcelwise_ttest.csv"
    df_ttest_reset.to_csv(outname, index=False)
    parcelwise_ttests[contrast_name] = df_ttest_reset
    print(f"Saved: {outname}")
    print(f"Top 5 parcels by p-value ({contrast_name}):")
    print(df_ttest.sort_values("pvalue").head(5))

In [ ]:
# ── NiSpace group-level colocalization (Hedges' g) ────────────────────────────
df_all, outputs = run_group_comparisons(
    Y_aligned=Y_aligned, design=design, ref_df=df_reference_selected,
    contrasts=CONTRASTS, labels=GROUP_LABELS, n_perm=N_PERM,
)
df_all.to_csv("../../results/nispace_group_comparison_results_subgroups_thickness.csv", index=False)
print("Saved group comparison results.")
display(df_all.head(10))

In [ ]:
# ── Single-subject z-score colocalization ─────────────────────────────────────
zscore_outputs = {}
for contrast_name, (gA, gB) in CONTRASTS.items():
    mask = design["CONCOHORT"].astype(float).isin([gA, gB])
    y = Y_aligned.loc[mask].copy()
    d = design.loc[mask].copy()
    keep = ~(y.isna().any(axis=1) | d[["age","SEX","Field Strength"]].isna().any(axis=1))
    y = y.loc[keep]; d = d.loc[keep]
    design_sub = build_nispace_design(d, gA, gB)
    print(f"\n{contrast_name}")
    print("Counts:", d["CONCOHORT"].astype(float).map(GROUP_LABELS).value_counts().to_dict())
    colocs, pvals, qvals, nsp = group_comparison(
        y=y, x=df_reference_selected, parcellation="DesikanKilliany", design=design_sub,
        comparison_method="zscore(a,b)", colocalization_method="spearman",
        n_perm=N_PERM, n_proc=-1, verbose=False, plot_design=False,
    )
    zscore_outputs[contrast_name] = {"colocs": colocs, "p": pvals, "q": qvals, "nsp": nsp}

In [ ]:
# ── Build merged dataframe for clinical correlation ───────────────────────────
all_rows = []
for contrast, res in zscore_outputs.items():
    colocs = res["colocs"].copy()
    if colocs.index.name is None:
        colocs.index.name = "PATNO"
    colocs_long = colocs.stack(list(range(colocs.columns.nlevels))).reset_index()
    colocs_long = colocs_long.rename(columns={0: "colocalization"})
    meta_cols = [c for c in colocs_long.columns if c not in ["PATNO", "colocalization"]]
    colocs_long["map"] = colocs_long[meta_cols].astype(str).agg(" | ".join, axis=1)
    colocs_long = colocs_long[["PATNO", "map", "colocalization"]].copy()
    colocs_long["contrast"] = contrast
    all_rows.append(colocs_long)

nispace_df = pd.concat(all_rows, ignore_index=True)

clinical_df = df[["PATNO", "updrs3_score", "moca", "gds", "SEX", "age"]].copy()
nispace_df["PATNO"] = nispace_df["PATNO"].astype(str)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)

merged_df = nispace_df.merge(clinical_df, on="PATNO", how="inner")
merged_df.to_csv("../../data/merged_df_thickness_cortical_subgroups.csv", index=False)
print("Saved merged_df_thickness_cortical_subgroups.csv")
print(merged_df.shape)
display(merged_df.head())